# Assignment 9B – Ensemble Learning: AdaBoost, GBM, XGBoost on Iris

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
df = pd.read_csv('Iris_d83f02.csv')
print(df.shape)
df.head(3)

In [ ]:
feature_cols = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
X = df[feature_cols].values
le = LabelEncoder()
y = le.fit_transform(df['Species'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

## Train & Evaluate Three Models

In [ ]:
models = {
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'GBM':      GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':  XGBClassifier(n_estimators=100, random_state=42,
                              use_label_encoder=False, eval_metric='mlogloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    cv   = cross_val_score(model, X, y, cv=5).mean()
    results[name] = {'Test Accuracy': round(acc, 4), 'CV Accuracy (5-fold)': round(cv, 4)}
    print(f'--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=le.classes_))

pd.DataFrame(results).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices – AdaBoost vs GBM vs XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy comparison bar chart
res_df = pd.DataFrame(results).T

plt.figure(figsize=(6, 4))
res_df.plot(kind='bar', ax=plt.gca(), color=['steelblue', 'seagreen'], edgecolor='k', linewidth=0.5)
plt.title('Model Accuracy Comparison – Iris')
plt.ylabel('Accuracy')
plt.ylim(0.9, 1.01)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()